# 19 — Advanced Python Review and Capstone Preparation

Integrate correctness, concurrency, reliability, security, observability, and architecture into one small offline service. This lesson summarizes the techniques needed for the final three portfolio projects.

*Advanced · 90–150 min*

## You will learn
- Combine typed records and explicit validation
- Use bounded asynchronous work and deterministic tests
- Describe idempotency and error handling at storage boundaries
- Apply measurement and reliability criteria

## Examples

### 1. Validated immutable events

Domain models capture a consistent contract. Frozen dataclasses are useful for event data, but type annotations need runtime validation and external data should be parsed at explicit boundaries.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Event:
    event_id: str
    category: str
    amount: int
    def __post_init__(self):
        if not self.event_id or not self.category or isinstance(self.amount, bool) or not isinstance(self.amount, int) or self.amount < 0:
            raise ValueError("invalid event")

event = Event("E1", "books", 3)
print(event)

**Review:** What other constraints would a financial event require?

### 2. Bounded asynchronous transformation

An asyncio semaphore limits simultaneous work. TaskGroup gives the collection a bounded lifecycle. Results should have predictable associations with input records instead of relying on completion ordering.

In [ ]:
import asyncio

async def process_events(events, limit=3):
    semaphore = asyncio.Semaphore(limit)
    async def one(event):
        async with semaphore:
            await asyncio.sleep(0)
            return (event.event_id, event.amount * 2)
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(one(event)) for event in events]
    return [task.result() for task in tasks]

events = [Event(f"E{i}", "books", i) for i in range(5)]
print(asyncio.run(process_events(events)))

**Review:** How should partial results be handled when a TaskGroup child fails?

### 3. Idempotent aggregation

Deduplicate a logical event by ID before applying side effects. For real multi-process systems, enforcement must occur in durable, transactional storage. A local set is sufficient only for a single-process teaching example.

In [ ]:
def aggregate_once(events):
    seen, totals = set(), {}
    for event in events:
        if event.event_id in seen:
            continue
        seen.add(event.event_id)
        totals[event.category] = totals.get(event.category, 0) + event.amount
    return totals

sample = [Event("A", "books", 2), Event("A", "books", 2), Event("B", "pens", 1)]
print(aggregate_once(sample))
assert aggregate_once(sample) == {"books": 2, "pens": 1}

**Review:** Why is a simple set insufficient when many workers process the same message concurrently?

### 4. Testing and report generation

A final project should have tests for invalid input, duplicate handling, cancellation, and error boundaries. Keep reporting separate from domain calculations so that tests do not depend on print formatting.

In [ ]:
import unittest

class ReviewTests(unittest.TestCase):
    def test_duplicate_is_ignored(self):
        self.assertEqual(aggregate_once(sample)["books"], 2)
    def test_bad_event(self):
        with self.assertRaises(ValueError):
            Event("", "books", 4)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ReviewTests)
assert unittest.TextTestRunner(verbosity=0).run(suite).wasSuccessful()

**Review:** Which important failure modes remain untested in these two cases?

## Exercises

**Exercise 1.** Build a bounded asynchronous ETL pipeline with a documented cancellation policy.

In [ ]:
# Exercise 1
# TODO: implement and test.

**Exercise 2.** Persist idempotency keys transactionally in SQLite and test duplicate deliveries.

In [ ]:
# Exercise 2
# TODO: implement and test.

**Exercise 3.** Add structured metrics for accepted, rejected, duplicated and timed-out events.

In [ ]:
# Exercise 3
# TODO: implement and test.

**Exercise 4.** Write an architecture decision record that explains three key trade-offs.

In [ ]:
# Exercise 4
# TODO: implement and test.

## Common mistakes

- A local set cannot enforce distributed idempotency.
- Cancellation and timeouts are not substitutes for rollback.
- Tests must cover error boundaries, not just the happy path.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Bounded concurrency means…

   - **a)** A cap on how much work runs at once
   - **b)** No limit on tasks
   - **c)** One task at a time always

**2.** Before optimizing, you should…

   - **a)** Measure
   - **b)** Rewrite in C
   - **c)** Add caching everywhere

**3.** A resilient service handles duplicate requests via…

   - **a)** Restarting
   - **b)** Idempotency
   - **c)** Ignoring them

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/19', {1: 'b9ec0359f1ca', 2: '9897f1ab70c5', 3: 'b25212ffb604'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** the [advanced projects](../projects/) and [exercise workbooks](../exercises/)